# 机端三相短路电流计算 — 论文算例复现

本 Notebook 复现 Wang et al. (2012) "From mathematical analysis to experimental calculation: 
teaching three-phase short-circuits of a synchronous generator" 中的算例。

**论文参数（Table 1）：**
| 参数 | 值 | 参数 | 值 |
|------|-----|------|-----|
| r | 0.0045 | Xd | 0.95 |
| X'd | 0.33 | X"d | 0.21 |
| Xq | 0.71 | X"q | 0.22 |
| T'd0 | 2800s | T"d0 | 30s |
| T"q0 | 68s | PF | 0.98 (leading) |

**三种方法对比：**
- `ia₁(t)` — 精确数学分析法（保留所有频率分量）
- `ia₂(t)` — 三段式实验法（工程常用）
- `ia₃(t)` — 工程简化法（最实用）

In [ ]:
# 导入所需库
import numpy as np
import matplotlib.pyplot as plt
from psa4teaching.shortcircuit.terminal_shortcircuit import (
    GeneratorSCParams,
    calculate_terminal_shortcircuit_mathematical,
    calculate_terminal_shortcircuit_experimental,
    calculate_terminal_shortcircuit_simplified,
    plot_comparison,
)

print("库导入成功！")

In [ ]:
# 设置论文参数（Table 1）
params = GeneratorSCParams(
    r=0.0045,
    Xd=0.95,           # d轴同步电抗
    Xd_prime=0.33,     # X'd（暂态电抗）
    Xd_doubleprime=0.21, # X"d（次暂态电抗）
    Xq=0.71,           # q轴同步电抗
    Xq_prime=0.71,      # X'q ≈ Xq
    Xq_doubleprime=0.22, # X"q
    Td0_prime=2800.0,   # T'd0 (s)
    Td0_doubleprime=30.0, # T"d0 (s)
    Tq0_prime=0.0,       # T'q0 (s)
    Tq0_doubleprime=68.0, # T"q0 (s)
    U0=1.0,              # 额定电压 (p.u.)
    PF=0.98,             # 功率因数（超前）
    omega=2*np.pi*50,   # 50Hz系统
)

print("论文参数设置完成！")
print(f"X'd = {params.Xd_prime}, X\"d = {params.Xd_doubleprime}")
print(f"T'd0 = {params.Td0_prime}s, T\"d0 = {params.Td0_doubleprime}s")

## 一、精确数学分析法 — ia₁(t)

对应论文第7~9页，保留所有频率分量（含 0.0005Hz 和 1.9995Hz）。
这是最精确的方法，但计算最复杂。

In [ ]:
# 计算 ia₁(t) — 精确数学法
t_end = 10.0   # 仿真10秒
dt = 0.001    # 步长1ms

result_math = calculate_terminal_shortcircuit_mathematical(
    params, t_end=t_end, dt=dt, theta0=0.0, verbose=True
)

print(f"\n计算完成！")
print(f"时间点数：{len(result_math.t)}")
print(f"ia 最大值：{np.max(np.abs(result_math.ia)):.4f} p.u.")
print(f"ia 稳态值：{np.abs(result_math.ia[-1]):.4f} p.u.")

In [ ]:
# 绘制 ia₁(t) 各分量（对应论文 Fig.3）
result_math.plot(
    show_components=True,
    t_range=(0, 1),          # 显示前1秒
    title="Fig.3 - ia₁(t) 各分量（精确数学法）"
)
plt.show()

## 二、三段式实验法 — ia₂(t)

对应论文第11~12页，将短路过程分为次暂态、暂态、稳态三个阶段。
工程中最常用的方法，精度与效率平衡良好。

In [ ]:
# 计算 ia₂(t) — 三段式实验法
result_exp = calculate_terminal_shortcircuit_experimental(
    params, t_end=t_end, dt=dt, theta0=0.0, verbose=True
)

print(f"\n计算完成！")
print(f"ia 最大值：{np.max(np.abs(result_exp.ia)):.4f} p.u.")
print(f"ia 稳态值：{np.abs(result_exp.ia[-1]):.4f} p.u.")

In [ ]:
# 绘制 ia₂(t) 各分量（对应论文 Fig.4）
result_exp.plot(
    show_components=True,
    t_range=(0, 1),
    title="Fig.4 - ia₂(t) 各分量（三段式实验法）"
)
plt.show()

## 三、工程简化法 — ia₃(t)

对应论文第13页，假设 X″d = X″q，消除倍频分量。
形式最简，适用于规划设计和不需要高精度的场景。

In [ ]:
# 计算 ia₃(t) — 工程简化法
result_simp = calculate_terminal_shortcircuit_simplified(
    params, t_end=t_end, dt=dt, theta0=0.0, verbose=True
)

print(f"\n计算完成！")
print(f"ia 交流包络：{result_simp.ac_envelope[0]:.4f} p.u.")
print(f"ia 直流分量：{np.abs(result_simp.dc_component[0]):.4f} p.u.")

In [ ]:
# 绘制 ia₃(t) 电流曲线（对应论文 Fig.5）
result_simp.plot(
    show_components=False,
    t_range=(0, 10),
    title="Fig.5 - ia₃(t) 有效值曲线（工程简化法）"
)
plt.show()

## 四、三种方法对比（论文 Fig.7/8/9）

In [ ]:
# 对比 ia₁(t) 和 ia₂(t) 总电流（对应论文 Fig.7）
plt.figure(figsize=(12, 6))
t = result_math.t
plt.plot(t, np.abs(result_math.ia), 'b-',  label='ia₁(t) - 精确数学法', linewidth=2)
plt.plot(t, np.abs(result_exp.ia), 'r--', label='ia₂(t) - 三段式实验法', linewidth=1.5)
plt.xlabel('时间 t (s)')
plt.ylabel('|ia(t)| (p.u.)')
plt.title('Fig.7 - ia₁(t) 与 ia₂(t) 总电流对比')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xlim(0, 5)
plt.savefig('fig7_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 对比三种方法的交流分量和直流分量（对应论文 Fig.8）
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 交流分量对比
ax1.plot(result_math.t, result_math.ac_envelope, 'b-', label='ia₁(t) 交流', linewidth=2)
ax1.plot(result_exp.t, result_exp.ac_envelope, 'r--', label='ia₂(t) 交流', linewidth=1.5)
ax1.plot(result_simp.t, result_simp.ac_envelope, 'g:.', label='ia₃(t) 交流', linewidth=1.5)
ax1.set_xlabel('时间 t (s)')
ax1.set_ylabel('交流包络 (p.u.)')
ax1.set_title('Fig.8(a) - 交流分量对比')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_xlim(0, 0.5)

# 直流分量对比
ax2.plot(result_math.t, np.abs(result_math.dc_component), 'b-', label='ia₁(t) 直流', linewidth=2)
ax2.plot(result_exp.t, np.abs(result_exp.dc_component), 'r--', label='ia₂(t) 直流', linewidth=1.5)
ax2.plot(result_simp.t, np.abs(result_simp.dc_component), 'g:.', label='ia₃(t) 直流', linewidth=1.5)
ax2.set_xlabel('时间 t (s)')
ax2.set_ylabel('直流分量 (p.u.)')
ax2.set_title('Fig.8(b) - 直流分量对比')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_xlim(0, 0.5)

plt.tight_layout()
plt.savefig('fig8_components.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 三种方法总电流对比（对应论文 Fig.9）
plt.figure(figsize=(12, 6))
t = result_math.t
plt.plot(t, np.abs(result_math.ia), 'b-',  label='ia₁(t) 精确数学法', linewidth=2)
plt.plot(t, np.abs(result_exp.ia), 'r--', label='ia₂(t) 三段式实验法', linewidth=1.5)
plt.plot(t, np.abs(result_simp.ia), 'g:',  label='ia₃(t) 工程简化法', linewidth=1.5)
plt.xlabel('时间 t (s)')
plt.ylabel('|ia(t)| (p.u.)')
plt.title('Fig.9 - 三种方法总电流对比')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xlim(0, 10)
plt.savefig('fig9_total.png', dpi=300, bbox_inches='tight')
plt.show()

## 五、误差分析（论文 Table 2）

In [ ]:
# 计算 ia₂(t) 相对于 ia₁(t) 的误差（对应论文 Table 2）
def compute_error(result_ref, result_test, t_start, t_end):
    """计算指定时间段内的误差"""
    mask = (result_ref.t >= t_start) & (result_ref.t <= t_end)
    ref_vals = np.abs(result_ref.ia[mask])
    test_vals = np.abs(result_test.ia[mask])
    abs_err = np.max(np.abs(ref_vals - test_vals))
    rel_err = np.mean(np.abs((ref_vals - test_vals) / ref_vals)) * 100
    return abs_err, rel_err

# 论文的三个时间段
periods = [
    (0, 0.06, "次暂态段 (0~0.06s)"),
    (0.06, 1.0, "暂态段 (0.06~1s)"),
    (1.0, 10.0, "稳态段 (>1s)"),
]

print(f"{'时段':<25s} {'最大绝对误差':>12s} {'平均相对误差':>12s}")
print("-" * 55)
for t_start, t_end, name in periods:
    abs_err, rel_err = compute_error(result_math, result_exp, t_start, t_end)
    print(f"{name:<25s} {abs_err:>12.6f} {rel_err:>11.2f}%")

print("\n论文 Table 2 参考值：")
print("次暂态段：最大绝对误差 0.0505 p.u., 相对误差 0.88%")
print("暂态段：  最大绝对误差 0.0690 p.u., 相对误差 2.06%")
print("稳态段：    最大绝对误差 0.0461 p.u., 相对误差 2.27%")

In [ ]:
# 全部图片汇总展示
fig, axes = plt.subplots(3, 2, figsize=(14, 12))

# Fig.3 - ia₁(t) 各分量
axes[0,0].plot(result_math.t, result_math.fundamental, 'b-', label='基频', linewidth=1.5)
axes[0,0].plot(result_math.t, result_math.dc_component, 'r-', label='直流', linewidth=1.5)
axes[0,0].plot(result_math.t, result_math.double_freq, 'g-', label='倍频', linewidth=1.5)
axes[0,0].set_title('Fig.3 - ia₁(t) 各分量')
axes[0,0].set_xlim(0, 0.5)
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3)

# Fig.4 - ia₂(t) 各分量
axes[0,1].plot(result_exp.t, result_exp.fundamental, 'b-', label='基频', linewidth=1.5)
axes[0,1].plot(result_exp.t, result_exp.dc_component, 'r-', label='直流', linewidth=1.5)
axes[0,1].plot(result_exp.t, result_exp.double_freq, 'g-', label='倍频', linewidth=1.5)
axes[0,1].set_title('Fig.4 - ia₂(t) 各分量')
axes[0,1].set_xlim(0, 0.5)
axes[0,1].legend()
axes[0,1].grid(True, alpha=0.3)

# Fig.5 - ia₃(t) 有效值曲线
ia3_env = np.sqrt(result_simp.ac_envelope**2 + result_simp.dc_component**2)
axes[1,0].plot(result_simp.t, ia3_env, 'b-', linewidth=2)
axes[1,0].set_title('Fig.5 - ia₃(t) 有效值曲线')
axes[1,0].set_xlim(0, 10)
axes[1,0].grid(True, alpha=0.3)

# Fig.7 - ia₁ vs ia₂ 总电流
axes[1,1].plot(result_math.t, np.abs(result_math.ia), 'b-', label='ia₁(t)', linewidth=2)
axes[1,1].plot(result_exp.t, np.abs(result_exp.ia), 'r--', label='ia₂(t)', linewidth=1.5)
axes[1,1].set_title('Fig.7 - ia₁(t) 与 ia₂(t) 对比')
axes[1,1].set_xlim(0, 5)
axes[1,1].legend()
axes[1,1].grid(True, alpha=0.3)

# Fig.9 - 三种方法总电流
axes[2,0].plot(result_math.t, np.abs(result_math.ia), 'b-', label='ia₁(t)', linewidth=2)
axes[2,0].plot(result_exp.t, np.abs(result_exp.ia), 'r--', label='ia₂(t)', linewidth=1.5)
axes[2,0].plot(result_simp.t, np.abs(result_simp.ia), 'g:', label='ia₃(t)', linewidth=1.5)
axes[2,0].set_title('Fig.9 - 三种方法总电流对比')
axes[2,0].set_xlim(0, 10)
axes[2,0].legend()
axes[2,0].grid(True, alpha=0.3)

# 空白子图
axes[2,1].axis('off')

plt.tight_layout()
plt.savefig('all_figures.png', dpi=300, bbox_inches='tight')
plt.show()
print("所有图片已保存！")
print("  - fig7_comparison.png")
print("  - fig8_components.png")
print("  - fig9_total.png")
print("  - all_figures.png")